### Task: Using this sample dataset, create or optimize data preprocessing pipeline using opencv and pytorch and include image mask technique in the pipeline and save the output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
data_path =  "/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017_SAMPLE"

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.utils as vutils

dataset_dir = "./images"
output_dir = "./processed"
img_size = (256, 256)
batch_size = 8
num_workers = 0
save_tensors = True
visualize_grid = True

os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "masks"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "tensors"), exist_ok=True)

In [ ]:
# Mask generation helpers
def mask_from_grabcut(image_bgr, iter_count=5, rect_scale=0.95):
    h, w = image_bgr.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    rect_w = int(w * rect_scale)
    rect_h = int(h * rect_scale)
    rect_x = max(1, (w - rect_w) // 2)
    rect_y = max(1, (h - rect_h) // 2)
    rect = (rect_x, rect_y, rect_w, rect_h)

    bgdModel = np.zeros((1, 65), np.float64)
    fgdModel = np.zeros((1, 65), np.float64)

    try:
        cv2.grabCut(image_bgr, mask, rect, bgdModel, fgdModel, iterCount=iter_count, mode=cv2.GC_INIT_WITH_RECT)
        grabcut_mask = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype('uint8')
        if grabcut_mask.sum() < 0.01 * h * w * 255:
            return mask_from_otsu(image_bgr)
        return grabcut_mask
    except Exception as e:
        return mask_from_otsu(image_bgr)


def mask_from_otsu(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th

def refine_mask(mask, kernel_size=7, iterations=2):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=iterations)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel, iterations=iterations)
    _, binary = cv2.threshold(opened, 127, 255, cv2.THRESH_BINARY)
    return binary

In [ ]:
# Preprocessing and transforms
img_transform = T.Compose([
    T.ToPILImage(),
    T.Resize(img_size, interpolation=Image.BILINEAR),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])
mask_transform = T.Compose([
    T.ToPILImage(),
    T.Resize(img_size, interpolation=Image.NEAREST),
    T.ToTensor(),
])


In [ ]:
# PyTorch Dataset
class ImageMaskDataset(Dataset):
    def __init__(self, root_dir, generate_masks=True, cache_masks=False):
        """
        root_dir: folder containing images (jpg/png...). Subfolders are ignored.
        generate_masks: if True, generate mask per image (grabcut + refine). If False, try to load mask files next to images.
        cache_masks: if True, after generating mask it will be saved in output_dir/masks and reused.
        """
        self.root_dir = root_dir
        self.paths = [os.path.join(root_dir, f) for f in sorted(os.listdir(root_dir))
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
        self.generate_masks = generate_masks
        self.cache_masks = cache_masks

    def __len__(self):
        return len(self.paths)

    def _load_image(self, path):
        img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise RuntimeError(f"Failed to read image {path}")
        return img_bgr

    def _get_mask_path(self, img_path):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        return os.path.join(output_dir, "masks", f"{basename}_mask.png")

    def _generate_mask(self, img_bgr, img_path):
        mask_path = self._get_mask_path(img_path)
        if self.cache_masks and os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            return mask
        mask = mask_from_grabcut(img_bgr)
        mask = refine_mask(mask, kernel_size=7, iterations=2)
        if self.cache_masks:
            cv2.imwrite(mask_path, mask)
        return mask

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img_bgr = self._load_image(img_path)
        if self.generate_masks:
            mask = self._generate_mask(img_bgr, img_path)
        else:
            mask_file = self._get_mask_path(img_path)
            if os.path.exists(mask_file):
                mask = cv2.imread(mask_file, cv2.IMREAD_GRAYSCALE)
            else:
                mask = mask_from_otsu(img_bgr)
                mask = refine_mask(mask)

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_t = img_transform(img_rgb)
        mask_t = mask_transform(mask)
        mask_t = (mask_t > 0.5).float()

        sample = {
            "img": img_t,
            "mask": mask_t,
            "path": img_path
        }
        return sample

In [ ]:
# Running the pipeline and saving outputs
def process_and_save(dataset, batch_size=8, num_workers=0):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    saved_examples = []
    for batch_idx, batch in enumerate(loader):
        imgs = batch["img"]
        masks = batch["mask"]
        paths = batch["path"]
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        imgs_unnorm = imgs.clone() * std + mean
        B = imgs.shape[0]

        for i in range(B):
            img_tensor = imgs_unnorm[i]
            mask_tensor = masks[i]
            path = paths[i]
            basename = os.path.splitext(os.path.basename(path))[0]
            img_save_path = os.path.join(output_dir, "images", f"{basename}_proc.png")
            vutils.save_image(img_tensor, img_save_path)
            mask_save_path = os.path.join(output_dir, "masks", f"{basename}_proc_mask.png")
            vutils.save_image(mask_tensor, mask_save_path)
            if save_tensors:
                tensor_save_path = os.path.join(output_dir, "tensors", f"{basename}.pt")
                torch.save({
                    "img": imgs[i],
                    "mask": masks[i],
                    "path": path
                }, tensor_save_path)

            saved_examples.append((img_save_path, mask_save_path))

        print(f"Processed batch {batch_idx+1}/{len(loader)}")

    if visualize_grid and len(saved_examples) > 0:
        grid_items = []
        max_examples = min(16, len(saved_examples))
        for i in range(max_examples):
            img_path, mask_path = saved_examples[i]
            img = Image.open(img_path).convert("RGB")
            mask = Image.open(mask_path).convert("L").convert("RGB")
            grid_items.append(T.ToTensor()(img))
            grid_items.append(T.ToTensor()(mask))
        if grid_items:
            grid = vutils.make_grid(grid_items, nrow=4, padding=2)
            grid_out = os.path.join(output_dir, "grid_preview.png")
            vutils.save_image(grid, grid_out)
            print(f"Saved grid preview to {grid_out}")


In [ ]:
if __name__ == "__main__":
    print("Building dataset from:", dataset_dir)
    ds = ImageMaskDataset(dataset_dir, generate_masks=True, cache_masks=True)
    if len(ds) == 0:
        raise SystemExit("No images found in dataset_dir. Put JPG/PNG images in the folder and try again.")
    process_and_save(ds, batch_size=batch_size, num_workers=num_workers)
    print("Done. Processed images, masks, and tensors saved to:", output_dir)